# 04 — Few-Shot Training (Phase B)

Prototypical Network training on FineBadminton labeled data
with SSL pre-trained encoder from Phase A.

**Player ordering:** `FineBadmintonDataset` automatically reorders each skeleton so
the **hitting player is at nodes 0–16** and the opponent at nodes 17–33,
using the `"hitter"` field ("top"/"bottom") from the annotations.
This ensures the model always sees the strategically relevant player first.

**Pipeline:**
1. Load SSL pre-trained encoder (feature-layer aware)
2. Load FineBadminton labeled data with enriched features (L0–L3), hitter-first
3. 5-fold stratified CV with episodic sampling
4. Train ProtoNet (+ optional fine-tuning of encoder)
5. Evaluate: macro-F1, per-class F1, confusion matrix
6. Alternative classifiers: k-NN, linear probe
7. Save best prototypes and results

In [ ]:
import os, sys, zipfile
from pathlib import Path

# ── Colab detection ───────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT   = Path('/content/drive/MyDrive/Baddiev2')
    PROJECT_PATH = Path('/content/Baddiev2')
    ZIP_PATH     = DRIVE_ROOT / 'baddiev2_colab.zip'

    if not (PROJECT_PATH / 'src').exists():
        print('Extracting project files...')
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(PROJECT_PATH)
        print(f'Extracted to {PROJECT_PATH}')
    else:
        print('Project already extracted.')

    sys.path.insert(0, str(PROJECT_PATH))
    os.chdir(PROJECT_PATH)

    # Override paths so outputs go to Drive (not ephemeral /content)
    import src.config as _cfg
    _cfg.MODELS_DIR            = DRIVE_ROOT / 'models'
    _cfg.RESULTS_DIR           = DRIVE_ROOT / 'results'
    _cfg.SS_SKELETONS_GDINO    = DRIVE_ROOT / 'datasets_preprocessing' / 'shuttleset_skeletons_gdino'
    _cfg.FB_SKELETONS_GDINO_V2 = DRIVE_ROOT / 'datasets_preprocessing' / 'finebadminton_skeletons_gdino_v2'
    _cfg.FB_ANNOTATIONS        = (
        DRIVE_ROOT / 'Datasets' / 'FineBadminton-dataset' / 'dataset' /
        'transformed_combined_rounds_output_en_evals_translated.json'
    )
    # ── ShuttleSet CSV annotations ────────────────────────────────────────
    _cfg.SS_CSV_ROOT  = DRIVE_ROOT / 'datasets' / 'ShuttleSet' / 'set'
    _cfg.SS_MATCH_CSV = _cfg.SS_CSV_ROOT / 'match.csv'
    _cfg.SS_SPLIT_JSON = PROJECT_PATH / 'datasets_preprocessing' / 'shuttleset_split.json'

    _cfg.MODELS_DIR.mkdir(parents=True, exist_ok=True)
    _cfg.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    print(f'Drive root   : {DRIVE_ROOT}')
    print(f'Models dir   : {_cfg.MODELS_DIR}')
    print(f'SS skeletons : {_cfg.SS_SKELETONS_GDINO}')
    print(f'FB skeletons : {_cfg.FB_SKELETONS_GDINO_V2}')
    print(f'SS CSV root  : {_cfg.SS_CSV_ROOT}')
    print(f'SS CSV exists: {_cfg.SS_CSV_ROOT.exists()}')
else:
    sys.path.insert(0, os.path.abspath('..'))
    DRIVE_ROOT = Path('..')
    print('Local run — using paths from config.py')

import torch
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from tqdm import tqdm
from src.config import *

device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Device: {device}')
print(f'MODELS_DIR: {MODELS_DIR}')


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # resolves relative to current CWD (set by Colab cell above)

import torch
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from pathlib import Path
from functools import partial

from src.config import *
from src.data.graph_builder import GraphBuilder
from src.data.dataset import FineBadmintonDataset, EpisodicSampler, collate_episode
from src.models.stgcn_model import STGCN
from src.models.transformer_encoder import SkeletonTransformer
from src.models.proto_net import PrototypicalNetwork

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {device}")

# --- Skeleton source ---
# FB_SKELETONS           : original YOLOv8 (top-2 by confidence, ~14% chair umpire contamination)
# FB_SKELETONS_GDINO_V2  : GDINO-guided, all rallies — use this
SKELETON_DIR = FB_SKELETONS_GDINO_V2
print(f"Skeleton dir: {SKELETON_DIR}")

## 1. Configuration

In [ ]:
cfg = get_config('fewshot')

# Feature layer — must match what was used for SSL pre-training
FEATURE_LAYER = 'L2'
feature_dim = FEATURE_DIMS[FEATURE_LAYER]
cfg.stgcn.in_channels = feature_dim

print(f"Feature layer: {FEATURE_LAYER} ({feature_dim} features/node)")
print(f"\nEncoder: ST-GCN")
print(f"  in_channels: {cfg.stgcn.in_channels}")
print(f"  num_nodes: {cfg.stgcn.num_nodes}")
print(f"  embedding_dim: {cfg.stgcn.embedding_dim}")
print(f"\nProtoNet:")
print(f"  n_way: {cfg.proto.n_way}")
print(f"  k_shot: {cfg.proto.k_shot}")
print(f"  n_query: {cfg.proto.n_query}")
print(f"  distance: {cfg.proto.distance}")
print(f"  episodes_per_epoch: {cfg.proto.episodes_per_epoch}")
print(f"  epochs: {cfg.proto.epochs}")
print(f"  fine_tune_encoder: {cfg.proto.fine_tune_encoder}")

## 2. Build Encoder and Load Pre-trained Weights

In [ ]:
from src.data.graph_builder import GraphBuilder
from src.models.stgcn_model import STGCN

graph_builder = GraphBuilder(
    use_inter_player=cfg.ablation.use_inter_player,
    single_player=cfg.ablation.single_player,
)
adjacency = graph_builder.build_adjacency().to(device)
print(f'Adjacency: {adjacency.shape}')

encoder = STGCN(
    in_channels=cfg.stgcn.in_channels, num_nodes=cfg.stgcn.num_nodes, adjacency=adjacency,
    num_layers=cfg.stgcn.num_layers, base_channels=cfg.stgcn.base_channels,
    embedding_dim=cfg.stgcn.embedding_dim, temporal_kernel=cfg.stgcn.temporal_kernel,
    dropout=cfg.stgcn.dropout,
).to(device)
print(f'Encoder parameters: {sum(p.numel() for p in encoder.parameters()):,}')

# Load SSL pre-trained weights — prefer supcon > simclr > legacy
ssl_checkpoint = None
SSL_METHOD_LOADED = None

if cfg.ablation.use_pretrained:
    for method in ('supcon', 'simclr'):
        candidate = MODELS_DIR / f'ssl_pretrained_{method}_{FEATURE_LAYER}.pt'
        if candidate.exists():
            ssl_checkpoint = torch.load(candidate, map_location=device, weights_only=False)
            encoder.load_state_dict(ssl_checkpoint['encoder_state_dict'])
            SSL_METHOD_LOADED = method
            print(f'Loaded SSL weights: {candidate.name}')
            print(f'  ssl_method: {ssl_checkpoint.get("ssl_method", method)}')
            if 'history' in ssl_checkpoint:
                print(f'  final loss: {ssl_checkpoint["history"][-1]:.4f}')
            break
    if ssl_checkpoint is None:
        legacy = MODELS_DIR / f'ssl_pretrained_{FEATURE_LAYER}.pt'
        if legacy.exists():
            ssl_checkpoint = torch.load(legacy, map_location=device, weights_only=False)
            encoder.load_state_dict(ssl_checkpoint['encoder_state_dict'])
            SSL_METHOD_LOADED = 'legacy'
            print(f'Loaded SSL weights (legacy): {legacy.name}')

if ssl_checkpoint is None:
    print('No SSL weights found — using random init')


In [ ]:
# Hit/shot type auxiliary head config (Phase B)
HIT_TYPE_AUX_WEIGHT = 0.3  # same weight as SSL aux loss in Phase A

# determine which auxiliary labels are available (FineBadminton vs. ShuttleSet)
if 'NUM_FB_HIT_TYPES' in globals():
    aux_label_name = 'FB hit types'
    aux_types = FB_HIT_TYPES
    num_aux_types = NUM_FB_HIT_TYPES
elif 'NUM_SS_SHOT_TYPES' in globals():
    aux_label_name = 'SS shot types'
    aux_types = SS_SHOT_TYPES
    num_aux_types = NUM_SS_SHOT_TYPES
else:
    aux_label_name = 'aux labels'
    aux_types = []
    num_aux_types = 0

print(f"{aux_label_name} ({num_aux_types} classes): {aux_types}")
print(f"Auxiliary loss weight: {HIT_TYPE_AUX_WEIGHT}")

## 3. Load FineBadminton Data

In [ ]:
dataset = FineBadmintonDataset(
    skeleton_dir=SKELETON_DIR,
    shot_window=cfg.data.shot_window,
    feature_layer=FEATURE_LAYER,
)
print(f"Dataset size: {len(dataset)}")

# Show class distribution
from collections import Counter
label_dist = Counter(dataset.get_labels())
print(f"\nClass distribution:")
for idx in sorted(label_dist.keys()):
    print(f"  {IDX_TO_STRATEGY[idx]}: {label_dist[idx]}")

# Show how many samples actually have skeleton files
has_skel = sum(1 for info in dataset.rally_info if info['has_skeleton'])
print(f"\nSamples with skeleton files: {has_skel}/{len(dataset)}")

# --- Rally-level splits (no data leakage) ---
# 30 train rallies / 10 held-out rallies (of 40 total)
N_TRAIN_RALLIES = 30
train_rally_idx, holdout_rally_idx = dataset.get_rally_splits(
    n_train_rallies=N_TRAIN_RALLIES,
    seed=cfg.data.random_seed,
)
print(f"\nRally-level splits:")
train_labels = [dataset.get_labels()[i] for i in train_rally_idx]
holdout_labels = [dataset.get_labels()[i] for i in holdout_rally_idx]
print(f"  Train  ({len(train_rally_idx)} shots): {Counter(IDX_TO_STRATEGY[l] for l in train_labels)}")
print(f"  Held-out ({len(holdout_rally_idx)} shots): {Counter(IDX_TO_STRATEGY[l] for l in holdout_labels)}")

# --- 5-fold CV on training rallies only ---
# Note: splits are shot-stratified within train_rally_idx
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=cfg.data.num_folds, shuffle=True, random_state=cfg.data.random_seed)
splits = []
for tv_idx, test_idx in skf.split(train_rally_idx, train_labels):
    tv_labels = [train_labels[i] for i in tv_idx]
    inner_skf = StratifiedKFold(n_splits=max(2, cfg.data.num_folds - 1),
                                 shuffle=True, random_state=cfg.data.random_seed)
    tr_idx, val_idx = next(inner_skf.split(tv_idx, tv_labels))
    splits.append((
        [train_rally_idx[i] for i in tv_idx[tr_idx]],
        [train_rally_idx[i] for i in tv_idx[val_idx]],
        [train_rally_idx[i] for i in test_idx],
    ))

print(f"\n5-Fold CV on training rallies:")
for i, (tr, val, te) in enumerate(splits):
    print(f"  Fold {i+1}: train={len(tr)}, val={len(val)}, test={len(te)}")

# Build hit/shot type label array for Phase B auxiliary loss
if 'FB_HIT_TYPE_TO_IDX' in globals():
    hit_type_labels_all = np.array([
        FB_HIT_TYPE_TO_IDX.get(info.get('hit_type', '').lower().strip(), -1)
        for info in dataset.rally_info
    ])
    aux_label_name = 'hit type'
elif 'SS_SHOT_TYPE_TO_IDX' in globals():
    hit_type_labels_all = np.array([
        SS_SHOT_TYPE_TO_IDX.get(info.get('shot_type', ''), -1)
        for info in dataset.rally_info
    ])
    aux_label_name = 'shot type'
else:
    hit_type_labels_all = np.array([-1] * len(dataset))
    aux_label_name = 'aux label'

valid_ht = (hit_type_labels_all >= 0).sum()
print(f"\n{aux_label_name.capitalize()} coverage: {valid_ht}/{len(dataset)} shots have {aux_label_name} labels")


## 4. Helper Functions

In [ ]:
def extract_embeddings(encoder, dataset, indices, device, batch_size=64):
    """Extract embeddings for a subset of the dataset (batched)."""
    encoder.eval()
    embeddings, labels = [], []
    with torch.no_grad():
        for start in range(0, len(indices), batch_size):
            batch_idx = indices[start:start + batch_size]
            xs, ys = zip(*[dataset[i] for i in batch_idx])
            xs = torch.stack(xs).to(device)
            emb = encoder(xs).cpu().numpy()
            embeddings.append(emb)
            labels.extend(ys)
    return np.concatenate(embeddings, axis=0), np.array(labels)

def train_one_epoch(encoder, proto_net, dataset, train_idx, optimizer, cfg, device,
                    hit_type_head=None, hit_type_labels_all=None, aux_weight=0.3):
    """Train one epoch of episodic training with optional hit type auxiliary loss."""
    encoder.train()
    if hit_type_head is not None:
        hit_type_head.train()
    train_labels = [dataset.get_labels()[i] for i in train_idx]

    sampler = EpisodicSampler(
        labels=train_labels,
        n_way=cfg.proto.n_way,
        k_shot=cfg.proto.k_shot,
        n_query=cfg.proto.n_query,
        episodes_per_epoch=cfg.proto.episodes_per_epoch,
    )

    per_class = cfg.proto.k_shot + cfg.proto.n_query
    epoch_loss, epoch_acc, n_episodes = 0.0, 0.0, 0

    for episode_indices in sampler:
        actual_indices = [train_idx[i] for i in episode_indices]

        batch = [dataset[i] for i in actual_indices]
        support_x, support_y, query_x, query_y = collate_episode(
            batch, cfg.proto.n_way, cfg.proto.k_shot, cfg.proto.n_query
        )

        support_x = support_x.to(device)
        support_y = support_y.to(device)
        query_x = query_x.to(device)
        query_y = query_y.to(device)

        support_emb = encoder(support_x)
        query_emb = encoder(query_x)

        n_way_actual = int(support_y.unique().numel())
        prototypes = proto_net.compute_prototypes(support_emb, support_y, n_way_actual)
        distances = proto_net.compute_distances(query_emb, prototypes)
        log_probs = torch.nn.functional.log_softmax(-distances, dim=1)
        proto_loss = torch.nn.functional.nll_loss(log_probs, query_y)

        # Hit type auxiliary loss
        # Reconstruct actual indices in collate_episode order (all support first, then all query)
        aux_loss = torch.tensor(0.0, device=device)
        if hit_type_head is not None and hit_type_labels_all is not None:
            n_way_ep = len(actual_indices) // per_class
            support_actual, query_actual = [], []
            for c in range(n_way_ep):
                base = c * per_class
                support_actual.extend(actual_indices[base:base + cfg.proto.k_shot])
                query_actual.extend(actual_indices[base + cfg.proto.k_shot:base + per_class])
            all_actual = support_actual + query_actual
            all_emb = torch.cat([support_emb, query_emb], dim=0)

            ht_labels = torch.tensor(
                [hit_type_labels_all[i] for i in all_actual],
                dtype=torch.long, device=device
            )
            valid_mask = ht_labels >= 0
            if valid_mask.sum() > 0:
                ht_logits = hit_type_head(all_emb[valid_mask])
                aux_loss = torch.nn.functional.cross_entropy(ht_logits, ht_labels[valid_mask])

        loss = proto_loss + aux_weight * aux_loss

        preds = (-distances).argmax(dim=1)
        acc = (preds == query_y).float().mean().item()

        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc
        n_episodes += 1

    return epoch_loss / n_episodes, epoch_acc / n_episodes


def evaluate_protonet(encoder, dataset, support_idx, test_idx, device):
    """Evaluate ProtoNet: compute prototypes from support, classify test."""
    support_emb, support_labels = extract_embeddings(encoder, dataset, support_idx, device)
    test_emb, test_labels = extract_embeddings(encoder, dataset, test_idx, device)

    prototypes = {}
    for c in np.unique(support_labels):
        prototypes[c] = support_emb[support_labels == c].mean(axis=0)

    preds = []
    for emb in test_emb:
        dists = {c: np.linalg.norm(emb - p) for c, p in prototypes.items()}
        preds.append(min(dists, key=dists.get))
    preds = np.array(preds)

    all_labels_list = list(range(NUM_CLASSES))
    macro_f1 = f1_score(test_labels, preds, average='macro',
                        labels=all_labels_list, zero_division=0)
    per_class = f1_score(test_labels, preds, average=None,
                         labels=all_labels_list, zero_division=0)
    cm = confusion_matrix(test_labels, preds, labels=all_labels_list)
    report = classification_report(
        test_labels, preds,
        labels=all_labels_list,
        target_names=STRATEGY_CLASSES,
        output_dict=True,
        zero_division=0,
    )

    return {
        'macro_f1': macro_f1,
        'per_class_f1': per_class.tolist(),
        'confusion_matrix': cm,
        'report': report,
        'prototypes': prototypes,
        'predictions': preds,
        'true_labels': test_labels,
    }


## 5. Episodic Training Loop (5-Fold CV)

In [ ]:
all_fold_results = []
training_history = []

for fold_idx, (train_idx, val_idx, test_idx) in enumerate(splits):
    print(f"\n{'='*60}")
    print(f"Fold {fold_idx + 1}/{len(splits)}")
    print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

    # Reset encoder to SSL pre-trained weights for each fold
    if ssl_checkpoint is not None:
        encoder.load_state_dict(ssl_checkpoint['encoder_state_dict'])
    else:
        encoder = STGCN(
            in_channels=cfg.stgcn.in_channels,
            num_nodes=cfg.stgcn.num_nodes,
            adjacency=adjacency,
            num_layers=cfg.stgcn.num_layers,
            base_channels=cfg.stgcn.base_channels,
            embedding_dim=cfg.stgcn.embedding_dim,
            temporal_kernel=cfg.stgcn.temporal_kernel,
            dropout=cfg.stgcn.dropout,
        ).to(device)

    # Reset hit/shot type head for each fold
    hit_type_head = torch.nn.Linear(cfg.stgcn.embedding_dim, num_aux_types).to(device)

    proto_net = PrototypicalNetwork(encoder, distance=cfg.proto.distance)

    if cfg.proto.fine_tune_encoder:
        optimizer = optim.Adam([
            {'params': encoder.parameters(), 'lr': cfg.proto.lr * cfg.proto.encoder_lr_scale},
            {'params': hit_type_head.parameters(), 'lr': cfg.proto.lr},
        ], weight_decay=1e-5)
    else:
        encoder.eval()
        for p in encoder.parameters():
            p.requires_grad = False
        optimizer = optim.Adam(hit_type_head.parameters(), lr=cfg.proto.lr, weight_decay=1e-5)

    fold_history = {'train_loss': [], 'train_acc': [], 'val_f1': []}
    best_val_f1 = 0.0
    best_encoder_state = None
    best_hit_type_head_state = None

    for epoch in range(cfg.proto.epochs):
        train_loss, train_acc = train_one_epoch(
            encoder, proto_net, dataset, train_idx, optimizer, cfg, device,
            hit_type_head=hit_type_head,
            hit_type_labels_all=hit_type_labels_all,
            aux_weight=HIT_TYPE_AUX_WEIGHT,
        )
        fold_history['train_loss'].append(train_loss)
        fold_history['train_acc'].append(train_acc)

        if (epoch + 1) % 5 == 0:
            val_result = evaluate_protonet(encoder, dataset, train_idx, val_idx, device)
            fold_history['val_f1'].append(val_result['macro_f1'])

            if val_result['macro_f1'] > best_val_f1:
                best_val_f1 = val_result['macro_f1']
                best_encoder_state = {k: v.clone() for k, v in encoder.state_dict().items()}
                best_hit_type_head_state = {k: v.clone() for k, v in hit_type_head.state_dict().items()}

            if (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1}/{cfg.proto.epochs} — "
                      f"Loss: {train_loss:.4f}, Acc: {train_acc:.3f}, "
                      f"Val-F1: {val_result['macro_f1']:.3f}")

    if best_encoder_state is not None:
        encoder.load_state_dict(best_encoder_state)
    if best_hit_type_head_state is not None:
        hit_type_head.load_state_dict(best_hit_type_head_state)

    test_result = evaluate_protonet(encoder, dataset, train_idx, test_idx, device)

    print(f"\n  Test Macro-F1: {test_result['macro_f1']:.3f}")
    print(f"  Per-class F1: {[f'{f:.3f}' for f in test_result['per_class_f1']]}")
    print(f"  Best Val-F1: {best_val_f1:.3f}")

    fold_result = {
        'fold': fold_idx,
        'macro_f1': test_result['macro_f1'],
        'per_class_f1': test_result['per_class_f1'],
        'confusion_matrix': test_result['confusion_matrix'],
        'report': test_result['report'],
        'best_val_f1': best_val_f1,
        'prototypes': test_result['prototypes'],
    }
    all_fold_results.append(fold_result)
    training_history.append(fold_history)

f1_scores = [r['macro_f1'] for r in all_fold_results]
print(f"\n{'='*60}")
print(f"Overall Macro-F1: {np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}")
print(f"Per-fold: {[f'{f:.3f}' for f in f1_scores]}")

---
## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, hist in enumerate(training_history):
    axes[0].plot(hist['train_loss'], alpha=0.7, label=f'Fold {i+1}')
    axes[1].plot(hist['train_acc'], alpha=0.7, label=f'Fold {i+1}')
    val_epochs = list(range(4, len(hist['train_loss']), 5))
    axes[2].plot(val_epochs, hist['val_f1'], 'o-', alpha=0.7, label=f'Fold {i+1}')

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Episodic Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Episode Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Macro-F1')
axes[2].set_title('Validation Macro-F1')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Few-Shot Training ({FEATURE_LAYER} features)', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fewshot_training_curves.png', dpi=150)
plt.show()

---
## 7. Confusion Matrix

In [ ]:
# Averaged confusion matrix across folds
avg_cm = np.mean([r['confusion_matrix'] for r in all_fold_results], axis=0)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    avg_cm, annot=True, fmt='.1f', cmap='Blues',
    xticklabels=STRATEGY_CLASSES,
    yticklabels=STRATEGY_CLASSES,
    ax=ax,
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Averaged Confusion Matrix (5-Fold)\nMacro-F1: {np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fewshot_confusion_matrix.png', dpi=150)
plt.show()

# Per-class F1 summary
per_class_f1s = np.array([r['per_class_f1'] for r in all_fold_results])
print("Per-class F1 (mean ± std):")
for i, name in enumerate(STRATEGY_CLASSES):
    if i < per_class_f1s.shape[1]:
        print(f"  {name:15s}: {per_class_f1s[:, i].mean():.3f} ± {per_class_f1s[:, i].std():.3f}")

---
## 8. Alternative Classifiers (k-NN, Linear Probe)

Compare ProtoNet against k-NN and logistic regression
using the same frozen encoder embeddings.

In [ ]:
# Use the best encoder from the last fold (or reload SSL weights)
if ssl_checkpoint is not None:
    encoder.load_state_dict(ssl_checkpoint['encoder_state_dict'])

# Extract all embeddings once
all_emb, all_labels = extract_embeddings(
    encoder, dataset, list(range(len(dataset))), device
)
print(f"Embeddings: {all_emb.shape}")

# Compare classifiers across folds
classifier_results = {
    'protonet': [],
    'knn_3': [],
    'knn_5': [],
    'linear_probe': [],
}

for fold_idx, (train_idx, val_idx, test_idx) in enumerate(splits):
    X_train = all_emb[train_idx]
    y_train = all_labels[train_idx]
    X_test = all_emb[test_idx]
    y_test = all_labels[test_idx]

    # ProtoNet (nearest centroid)
    prototypes = {}
    for c in np.unique(y_train):
        prototypes[c] = X_train[y_train == c].mean(axis=0)
    proto_preds = np.array([
        min(prototypes, key=lambda c: np.linalg.norm(x - prototypes[c]))
        for x in X_test
    ])
    classifier_results['protonet'].append(
        f1_score(y_test, proto_preds, average='macro')
    )

    # k-NN (k=3)
    knn3 = KNeighborsClassifier(n_neighbors=3)
    knn3.fit(X_train, y_train)
    classifier_results['knn_3'].append(
        f1_score(y_test, knn3.predict(X_test), average='macro')
    )

    # k-NN (k=5)
    knn5 = KNeighborsClassifier(n_neighbors=5)
    knn5.fit(X_train, y_train)
    classifier_results['knn_5'].append(
        f1_score(y_test, knn5.predict(X_test), average='macro')
    )

    # Linear probe (logistic regression)
    lr = LogisticRegression(max_iter=1000, C=1.0)
    lr.fit(X_train, y_train)
    classifier_results['linear_probe'].append(
        f1_score(y_test, lr.predict(X_test), average='macro')
    )

# Summary table
print(f"\n{'Method':<15} {'Macro-F1':>12} {'Std':>8}")
print("-" * 38)
for method, f1s in classifier_results.items():
    print(f"{method:<15} {np.mean(f1s):>12.3f} {np.std(f1s):>8.3f}")

---
## 9. Save Results and Best Model

In [ ]:
# Save best fold prototypes and encoder for inference
best_fold = max(range(len(all_fold_results)), key=lambda i: all_fold_results[i]['macro_f1'])
best_result = all_fold_results[best_fold]
print(f"Best fold: {best_fold + 1} (Macro-F1: {best_result['macro_f1']:.3f})")

proto_tensors = {c: torch.tensor(p) for c, p in best_result['prototypes'].items()}

checkpoint = {
    'encoder_state_dict': encoder.state_dict(),
    'hit_type_head_state_dict': hit_type_head.state_dict(),
    'prototypes': proto_tensors,
    'feature_layer': FEATURE_LAYER,
    'hit_types': aux_types,
    'config': cfg,
    'fold_results': [
        {k: v.tolist() if isinstance(v, np.ndarray) else v
         for k, v in r.items() if k != 'prototypes'}
        for r in all_fold_results
    ],
    'classifier_comparison': {
        method: {'mean': float(np.mean(f1s)), 'std': float(np.std(f1s)), 'folds': f1s}
        for method, f1s in classifier_results.items()
    },
}

save_path = MODELS_DIR / f'fewshot_{FEATURE_LAYER}.pt'
torch.save(checkpoint, save_path)
print(f"Saved to {save_path}")

json_results = {
    'feature_layer': FEATURE_LAYER,
    'overall_macro_f1': float(np.mean(f1_scores)),
    'overall_std': float(np.std(f1_scores)),
    'fold_f1s': [float(f) for f in f1_scores],
    'classifier_comparison': checkpoint['classifier_comparison'],
}
with open(RESULTS_DIR / 'fewshot_results.json', 'w') as f:
    json.dump(json_results, f, indent=2)
print(f"Results saved to {RESULTS_DIR / 'fewshot_results.json'}")

---
## 10. Held-Out Rally Evaluation

Final evaluation on the **10 held-out rallies** (completely unseen during CV).
Train prototypes on all 30 training-rally shots, evaluate on held-out shots.

This is the honest generalization estimate — no rally appears in both train and test.

In [ ]:
# Reload best SSL weights for final training run
if ssl_checkpoint is not None:
    encoder.load_state_dict(ssl_checkpoint['encoder_state_dict'])
else:
    encoder = STGCN(
        in_channels=cfg.stgcn.in_channels,
        num_nodes=cfg.stgcn.num_nodes,
        adjacency=adjacency,
        num_layers=cfg.stgcn.num_layers,
        base_channels=cfg.stgcn.base_channels,
        embedding_dim=cfg.stgcn.embedding_dim,
        temporal_kernel=cfg.stgcn.temporal_kernel,
        dropout=cfg.stgcn.dropout,
    ).to(device)

hit_type_head = torch.nn.Linear(cfg.stgcn.embedding_dim, num_aux_types).to(device)
proto_net = PrototypicalNetwork(encoder, distance=cfg.proto.distance)

if cfg.proto.fine_tune_encoder:
    optimizer = optim.Adam([
        {'params': encoder.parameters(), 'lr': cfg.proto.lr * cfg.proto.encoder_lr_scale},
        {'params': hit_type_head.parameters(), 'lr': cfg.proto.lr},
    ], weight_decay=1e-5)
else:
    for p in encoder.parameters():
        p.requires_grad = False
    optimizer = optim.Adam(hit_type_head.parameters(), lr=cfg.proto.lr, weight_decay=1e-5)

# Train on all 30 training-rally shots
print(f"Training on {len(train_rally_idx)} shots from {N_TRAIN_RALLIES} rallies...")
best_val_f1 = 0.0
best_encoder_state = None

# Use a held-back portion of train for validation (inner split)
inner_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=cfg.data.random_seed)
final_train_tv_idx, final_val_idx = next(
    inner_skf.split(train_rally_idx, train_labels)
)
final_train_idx = [train_rally_idx[i] for i in final_train_tv_idx]
final_val_idx_abs = [train_rally_idx[i] for i in final_val_idx]

for epoch in range(cfg.proto.epochs):
    train_loss, train_acc = train_one_epoch(
        encoder, proto_net, dataset, final_train_idx, optimizer, cfg, device,
        hit_type_head=hit_type_head,
        hit_type_labels_all=hit_type_labels_all,
        aux_weight=HIT_TYPE_AUX_WEIGHT,
    )
    if (epoch + 1) % 5 == 0:
        val_result = evaluate_protonet(encoder, dataset, final_train_idx, final_val_idx_abs, device)
        if val_result['macro_f1'] > best_val_f1:
            best_val_f1 = val_result['macro_f1']
            best_encoder_state = {k: v.clone() for k, v in encoder.state_dict().items()}
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{cfg.proto.epochs} — "
                  f"Loss: {train_loss:.4f}, Val-F1: {val_result['macro_f1']:.3f}")

if best_encoder_state is not None:
    encoder.load_state_dict(best_encoder_state)

# Evaluate on held-out rallies
holdout_result = evaluate_protonet(encoder, dataset, train_rally_idx, holdout_rally_idx, device)

print(f"\n{'='*60}")
print(f"HELD-OUT RALLY EVALUATION")
print(f"  Train rallies: {N_TRAIN_RALLIES} ({len(train_rally_idx)} shots)")
print(f"  Held-out rallies: {40 - N_TRAIN_RALLIES} ({len(holdout_rally_idx)} shots)")
print(f"  Macro-F1: {holdout_result['macro_f1']:.3f}")
print(f"  Per-class F1:")
for i, (name, f1) in enumerate(zip(STRATEGY_CLASSES, holdout_result['per_class_f1'])):
    print(f"    {name:15s}: {f1:.3f}")
print(f"\n  CV Macro-F1 (train rallies): {np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}")
print(f"  Held-out Macro-F1:           {holdout_result['macro_f1']:.3f}")

# Confusion matrix for held-out set
fig, ax = plt.subplots(figsize=(8, 6))
import seaborn as sns
sns.heatmap(
    holdout_result['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
    xticklabels=STRATEGY_CLASSES, yticklabels=STRATEGY_CLASSES, ax=ax,
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Held-Out Rally Confusion Matrix\nMacro-F1: {holdout_result["macro_f1"]:.3f}')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fewshot_holdout_confusion_matrix.png', dpi=150)
plt.show()

# Save held-out results
holdout_json = {
    'feature_layer': FEATURE_LAYER,
    'n_train_rallies': N_TRAIN_RALLIES,
    'n_holdout_rallies': 40 - N_TRAIN_RALLIES,
    'train_shots': len(train_rally_idx),
    'holdout_shots': len(holdout_rally_idx),
    'holdout_macro_f1': float(holdout_result['macro_f1']),
    'holdout_per_class_f1': holdout_result['per_class_f1'],
    'cv_macro_f1_mean': float(np.mean(f1_scores)),
    'cv_macro_f1_std': float(np.std(f1_scores)),
}
with open(RESULTS_DIR / 'fewshot_holdout_results.json', 'w') as f:
    json.dump(holdout_json, f, indent=2)
print(f"\nSaved to {RESULTS_DIR / 'fewshot_holdout_results.json'}")
